# ApexPlanet Task 2 — EDA & Business Intelligence

Run cells from top to bottom. Upload the cleaned CSV to Colab or change the file path.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
import os

os.makedirs('results', exist_ok=True)
os.makedirs('charts', exist_ok=True)


In [ ]:
from google.colab import files
uploaded = files.upload()
# Use the exact uploaded filename below if it differs.


In [ ]:
filename = 'ApexPlanet_Task1_Cleaned_Dataset.csv'  # update if needed
df = pd.read_csv(filename)
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')
print('Shape:', df.shape)
display(df.head())


In [ ]:
print('Columns:', df.columns.tolist())
df.info()
print('\nMissing values:\n', df.isnull().sum())
print('\nExact duplicate rows:', df.duplicated().sum())
display(df.describe(include='all').T)


In [ ]:
overview = pd.DataFrame({'Metric':['Total rows','Total columns','Exact duplicate rows','Total missing cells','Unique customers','Unique orders','Total sales'], 'Value':[len(df), df.shape[1], int(df.duplicated().sum()), int(df.isnull().sum().sum()), df['Customer_ID'].nunique(), df['Order_ID'].nunique(), round(df['Total_Sales'].sum(),2)]})
display(overview)
overview.to_csv('results/dataset_overview.csv', index=False)
pd.DataFrame({'Column':df.columns,'Data_Type':df.dtypes.astype(str).values,'Missing_Values':df.isnull().sum().values}).to_csv('results/data_quality_report.csv', index=False)
df.describe(include='all').T.to_csv('results/descriptive_statistics.csv')


In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
for col in numeric_cols[:6]:
    plt.figure(figsize=(8,5))
    plt.hist(df[col].dropna(), bins=20, edgecolor='black')
    plt.title(f'Distribution of {col}')
    plt.xlabel(col); plt.ylabel('Frequency'); plt.tight_layout()
    plt.savefig(f'charts/{col}_histogram.png', dpi=150)
    plt.show(); plt.close()


In [ ]:
cat_cols = df.select_dtypes(include=['object','category','bool']).columns
for col in cat_cols[:5]:
    counts = df[col].fillna('Missing').astype(str).value_counts().head(10)
    plt.figure(figsize=(9,5)); counts.plot(kind='bar')
    plt.title(f'Top categories in {col}'); plt.xlabel(col); plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right'); plt.tight_layout()
    plt.savefig(f'charts/{col}_bar_chart.png', dpi=150)
    plt.show(); plt.close()


In [ ]:
numeric_df = df.select_dtypes(include=np.number)
if numeric_df.shape[1] >= 2:
    corr = numeric_df.corr()
    display(corr)
    plt.figure(figsize=(10,7)); plt.imshow(corr, vmin=-1, vmax=1, aspect='auto'); plt.colorbar(label='Correlation')
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=70, ha='right', fontsize=7)
    plt.yticks(range(len(corr.columns)), corr.columns, fontsize=7)
    plt.title('Correlation heatmap'); plt.tight_layout(); plt.savefig('charts/correlation_heatmap.png', dpi=150); plt.show(); plt.close()
    corr.to_csv('results/correlation_matrix.csv')


In [ ]:
conn = sqlite3.connect('results/analysis.db')
df_sql = df.copy()
df_sql['Order_Date'] = pd.to_datetime(df_sql['Order_Date'], errors='coerce').dt.strftime('%Y-%m-%d')
df_sql.to_sql('dataset', conn, if_exists='replace', index=False)
query = '''SELECT Category, COUNT(*) AS order_rows, ROUND(SUM(Total_Sales), 2) AS total_sales FROM dataset GROUP BY Category ORDER BY total_sales DESC;'''
display(pd.read_sql_query(query, conn))
# See business_questions.sql for all seven prepared queries.
conn.close()


## Interpretation checklist
- Describe what the histograms and bar charts show.
- Report the top category/product/city from SQL results.
- Explain correlation without claiming causation.
- Verify repeated Original_Order_ID values and outlier flags with the data owner.